# Transfer Capability

Companion notebook for Tutorial 04. Compare ATC and voltage-stress changes
between base and stressed cases on the IEEE 30-bus system.

We use case30 because it has thermal ratings on every branch, giving real
binding constraints—not the `inf` you get from unrated networks like case118.

## Islanding vs Operational Contingencies

Case30 has 3 bridge branches whose outage islands part of the network.
Including them as contingencies drives ATC to zero. Excluding islanding
contingencies from the credible set gives meaningful results. We show both.

In [ ]:
import surge
import numpy as np

base = surge.case30()
stressed = surge.case30()
stressed.scale_loads(1.10)

path = surge.transfer.TransferPath("area1_to_area3", [1], [30])

base_study = surge.transfer.prepare_transfer_study(base)
stressed_study = surge.transfer.prepare_transfer_study(stressed)

# Identify bridge branches via infinite LODF diagonal
lodf_mat = surge.dc.compute_lodf_matrix(base)
bridge_indices = [
    i for i in range(base.n_branches)
    if np.isinf(lodf_mat.lodf[i, i]) or np.isnan(lodf_mat.lodf[i, i])
]
non_bridge = [i for i in range(base.n_branches) if i not in bridge_indices]
print(f"Bridge branches (islanding): {bridge_indices}")

# All contingencies (islanding drives ATC to 0)
all_ctg = surge.transfer.AtcOptions(
    monitored_branches=list(range(base.n_branches)),
    contingency_branches=list(range(base.n_branches)),
)
base_all = base_study.compute_nerc_atc(path, all_ctg)

# Operational set: exclude bridge contingencies
ops_ctg = surge.transfer.AtcOptions(
    monitored_branches=list(range(base.n_branches)),
    contingency_branches=non_bridge,
)
base_ops = base_study.compute_nerc_atc(path, ops_ctg)
stressed_ops = stressed_study.compute_nerc_atc(path, ops_ctg)

base_stress = surge.contingency.compute_voltage_stress(base)
stressed_stress = surge.contingency.compute_voltage_stress(stressed)

print(f"\n{'':25s} {'Base':>10s} {'Stressed':>10s}")
print(f"{'ATC all ctg (MW)':25s} {base_all.atc_mw:10.1f}")
print(f"{'ATC ops ctg (MW)':25s} {base_ops.atc_mw:10.1f} {stressed_ops.atc_mw:10.1f}")
print(f"{'TTC (MW)':25s} {base_ops.ttc_mw:10.1f} {stressed_ops.ttc_mw:10.1f}")
print(f"{'Binding branch':25s} {str(base_ops.binding_branch):>10s} {str(stressed_ops.binding_branch):>10s}")
print(f"{'Binding contingency':25s} {str(base_ops.binding_contingency):>10s} {str(stressed_ops.binding_contingency):>10s}")
print(f"{'Max L-index':25s} {base_stress.max_l_index:10.4f} {stressed_stress.max_l_index:10.4f}")

## Multiple Transfer Paths

Compare the full and operational contingency sets across several paths.

In [ ]:
paths = [
    surge.transfer.TransferPath("area1_to_area3", [1], [30]),
    surge.transfer.TransferPath("gen2_to_load26", [2], [26]),
    surge.transfer.TransferPath("gen1_to_load24", [1], [24]),
]

print(f"{'Path':20s} {'All ctg':>10s} {'Ops ctg':>10s} {'Binding br':>12s} {'Ctg':>6s}")
for p in paths:
    ra = base_study.compute_nerc_atc(p, all_ctg)
    ro = base_study.compute_nerc_atc(p, ops_ctg)
    bb = str(ro.binding_branch) if ro.binding_branch is not None else "-"
    bc = str(ro.binding_contingency) if ro.binding_contingency is not None else "N-0"
    print(f"{p.name:20s} {ra.atc_mw:10.1f} {ro.atc_mw:10.1f} {bb:>12s} {bc:>6s}")